# Barkley pipe flow - space-time diagrams & the $r_c$ transition

Co-moving $x$-$t$ diagrams of the three regimes, and a quantitative sweep of the
front-expansion rate across $r$ showing the excitable->bistable transition at
$r_c\approx0.833$.


In [ ]:
# --- environment setup (works locally and on Google Colab) ---
try:
    import barkley_pipe  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/yuvrajdabhi2212/barkley-pipe-flow.git"], check=True)
import matplotlib.pyplot as plt
import numpy as np

## Space-time diagrams (co-moving frames)

In [ ]:
from barkley_pipe.continuous import PeriodicGrid, ContinuousParams, puff_seed, simulate
from barkley_pipe.diagnostics import front_kinematics
from barkley_pipe.plotting import plot_spacetime

def run(r, L, n, T, snaps=121):
    g = PeriodicGrid(n=n, length=L)
    return simulate(g, ContinuousParams(r=r),
                    puff_seed(g, center=L/2, width=5, q_amplitude=1.0, u_dip=0.6),
                    (0.0, T), n_snapshots=snaps)

runs = [(run(0.7, 400, 1600, 300), 0.7, "puff"),
        (run(0.85, 600, 2000, 300), 0.85, "weak slug"),
        (run(1.0, 800, 2400, 250), 1.0, "slug")]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (res, r, name) in zip(axes, runs):
    drift = front_kinematics(res.q, res.x, res.t, res.grid.length)["drift"]
    plot_spacetime(res, comoving_speed=drift, ax=ax); ax.set_title(f"{name}, r={r}")
plt.tight_layout(); plt.show()

Vertical band (puff) -> narrow cone (weak slug) -> wide cone (slug).

## The $r_c$ transition: front-expansion rate vs $r$

In [ ]:
from barkley_pipe.nullclines import critical_r

r_values = [0.60, 0.70, 0.78, 0.82, 0.833, 0.85, 0.90, 0.96, 1.00, 1.10]
expansion = []
for r in r_values:
    res = run(r, 500, 1250, 220, snaps=111)
    expansion.append(front_kinematics(res.q, res.x, res.t, res.grid.length)["expansion_rate"])

rc = critical_r()
plt.figure(figsize=(7, 5))
plt.axhline(0, color="0.7", lw=1); plt.axvline(rc, color="C2", ls="--", label=f"r_c={rc:.3f}")
plt.plot(r_values, expansion, "o-", color="#c1272d", lw=2)
plt.xlabel("r"); plt.ylabel("front expansion rate"); plt.legend(); plt.show()
print("expansion turns on at r_c =", round(rc, 3))

The expansion rate is ~0 for $r<r_c$ (equilibrium puffs) and turns on at $r_c\approx0.833$, growing monotonically into the slug regime.